# Lab Nessie CI/CD Data — Jour 2

**Profil docker :** `cicd` · **Durée :** 70 min · **Format :** Trio rôlé

---

## Le paradigme Git for Data

Nessie apporte aux données les mêmes workflows que Git apporte au code :

| Git (code) | Nessie (données) |
|------------|------------------|
| `git checkout -b feat/ma-feature` | Créer une branche Nessie |
| Modifier des fichiers | Modifier des tables Iceberg |
| `git diff` | Comparer les snapshots |
| Pull Request + CI | Valider avec dbt tests |
| `git merge main` | Merger la branche Nessie |
| `git revert` | Rollback vers un snapshot |

---

## Contexte — Incident RetailCo

> Un data engineer a déployé un pipeline dbt sur `main` qui recalcule `total_amount`
> avec un bug (il divise par 1.20 au lieu de multiplier).
> Les données Gold sont corrompues depuis 2 heures.
> Votre mission : identifier le commit Nessie qui a introduit le bug, rollback, corriger sur une branche, merger proprement.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import requests, json, subprocess, time

spark = (
    SparkSession.builder
    .appName('RetailCo-Nessie-CICD')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
    .getOrCreate()
)

NESSIE_API = 'http://nessie:19120/api/v2'

def nessie(method, path, data=None):
    url = f'{NESSIE_API}{path}'
    r = requests.request(method, url,
        headers={'Content-Type': 'application/json'},
        json=data)
    return r.json()

print(f'✅ Spark {spark.version} · Nessie API {NESSIE_API}')

# Branches disponibles
branches = nessie('GET', '/trees')
print('\n📋 Branches Nessie disponibles :')
for ref in branches.get('references', []):
    print(f"   {ref['type']:8} {ref['name']:30} → {ref.get('hash','')[:12]}...")

---
## ⏱️ Kata d'amorce (10 min) — Première branche Nessie

In [ ]:
# ── Créer une branche depuis main ─────────────────────────────────────────────
main_hash = nessie('GET', '/trees/main')['reference']['hash']
print(f'Hash main actuel : {main_hash[:16]}...')

# Créer la branche feat/nessie-demo
resp = nessie('POST', '/trees', {
    'name': 'feat/nessie-demo',
    'type': 'BRANCH',
    'hash': main_hash
})
print(f'✅ Branche créée : {resp.get("name", "feat/nessie-demo")}')

# Écrire dans la branche
spark.conf.set('spark.sql.catalog.nessie.ref', 'feat/nessie-demo')
spark.sql('CREATE NAMESPACE IF NOT EXISTS nessie.retailco')
spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.retailco.orders_test
    (order_id STRING, amount DOUBLE, ts TIMESTAMP)
    USING iceberg
""")
spark.sql("INSERT INTO nessie.retailco.orders_test VALUES ('o-1', 99.99, current_timestamp())")

# Vérifier que main n'est pas affecté
spark.conf.set('spark.sql.catalog.nessie.ref', 'main')
try:
    spark.sql('SELECT * FROM nessie.retailco.orders_test').show()
    print('⚠️ La table est visible sur main — inattendu')
except:
    print('✅ La table orders_test est invisible sur main — isolation confirmée !')
print('\n💡 Chaque branche a sa propre vue des tables — isolation complète.')

# Retour sur main
spark.conf.set('spark.sql.catalog.nessie.ref', 'main')

---
## ⏱️ Incident à résoudre (45 min)

### Partie A — Simuler l'incident (5 min)

In [ ]:
# ── Simulation de l'incident ──────────────────────────────────────────────────
# 1. Créer la table Silver correcte sur main
spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.retailco.silver_transactions
    (transaction_id STRING, store_region STRING, category STRING,
     total_amount DOUBLE, total_ttc DOUBLE, processed_ts TIMESTAMP)
    USING iceberg
""")

spark.sql("""
    INSERT INTO nessie.retailco.silver_transactions VALUES
    ('txn-001', 'Île-de-France', 'Électronique', 500.0, 600.0, current_timestamp()),
    ('txn-002', 'PACA',         'Vêtements',    100.0, 120.0, current_timestamp()),
    ('txn-003', 'Bretagne',     'Maison',        80.0,  96.0, current_timestamp())
""")

v_correct = nessie('GET', '/trees/main')['reference']['hash']
print(f'✅ État correct — hash: {v_correct[:16]}...')
print('   total_ttc = total_amount × 1.20 (correct)')

# 2. Appliquer le bug (divise au lieu de multiplier)
spark.sql("""
    INSERT INTO nessie.retailco.silver_transactions
    SELECT transaction_id, store_region, category,
           total_amount,
           ROUND(total_amount / 1.20, 2) AS total_ttc,  -- BUG: / au lieu de ×
           current_timestamp()
    FROM nessie.retailco.silver_transactions
""")

v_bug = nessie('GET', '/trees/main')['reference']['hash']
print(f'\n🐛 Bug introduit  — hash: {v_bug[:16]}...')
print('   total_ttc = total_amount / 1.20 (BUG !)')
print('\nValeurs actuelles (corrompues) :')
spark.sql('SELECT transaction_id, total_amount, total_ttc, total_ttc/total_amount AS ratio FROM nessie.retailco.silver_transactions').show()

### Partie B — Diagnostiquer avec le log Nessie (10 min)

**TODO 1** : Utilisez l'API Nessie pour trouver le commit qui a introduit le bug.

In [ ]:
# TODO 1 — Diagnostiquer
# 📝 Listez les commits sur main. Identifiez celui qui a introduit le bug.
#    Comparez les snapshots avant/après pour confirmer la corruption.

print('💡 nessie("GET", "/trees/main/log") pour le log des commits')
print('💡 nessie("GET", "/trees/main/entries") pour les entrées du catalog')
print('💡 spark.sql("SELECT * FROM nessie.retailco.silver_transactions.snapshots") pour les snapshots Iceberg')

In [ ]:
# ✅ SOLUTION TODO 1

log = nessie('GET', '/trees/main/log')
print('📋 Log Nessie (commits récents) :')
for entry in log.get('logEntries', [])[:5]:
    meta = entry.get('commitMeta', {})
    print(f"  {entry.get('commitMeta', {}).get('hash','')[:12]}  {meta.get('message','')[:50]}")

# Snapshots Iceberg
print('\n📋 Snapshots Iceberg de silver_transactions :')
spark.sql('SELECT snapshot_id, committed_at, operation, summary FROM nessie.retailco.silver_transactions.snapshots').show(truncate=False)

# Vérifier les données sur le snapshot correct (avant le bug)
print(f'\n🔍 Données sur le hash correct ({v_correct[:12]}...) :')
spark.conf.set('spark.sql.catalog.nessie.ref', v_correct)
spark.sql('SELECT transaction_id, total_amount, total_ttc, ROUND(total_ttc/total_amount,2) AS ratio FROM nessie.retailco.silver_transactions').show()
spark.conf.set('spark.sql.catalog.nessie.ref', 'main')

### Partie C — Rollback + correction sur branche (20 min)

**TODO 2** : Créez une branche `fix/silver-ttc-correction`, corrigez le bug, validez avec un test, mergez sur main.

In [ ]:
# TODO 2 — Fix sur branche + merge
# 📝 a) Rollback main vers le hash correct (v_correct)
#    b) Créer la branche fix/silver-ttc-correction
#    c) Appliquer la correction (× 1.20)
#    d) Valider : aucune ligne où total_ttc < total_amount
#    e) Merger sur main

print('💡 nessie("POST", "/trees/main/assign", {"hash": v_correct}) pour rollback')
print('💡 Merge : nessie("POST", "/trees/main/merge", {"fromRefName": "fix/...", "fromHash": ...})')

In [ ]:
# ✅ SOLUTION TODO 2

# 1. Rollback main vers l'état correct
nessie('POST', '/trees/main/assign', {'hash': v_correct})
print(f'✅ Rollback main → {v_correct[:12]}...')

# 2. Créer la branche de fix
current_hash = nessie('GET', '/trees/main')['reference']['hash']
nessie('POST', '/trees', {'name': 'fix/silver-ttc-correction', 'type': 'BRANCH', 'hash': current_hash})
print('✅ Branche fix/silver-ttc-correction créée')

# 3. Travailler sur la branche de fix
spark.conf.set('spark.sql.catalog.nessie.ref', 'fix/silver-ttc-correction')
spark.sql("""
    INSERT INTO nessie.retailco.silver_transactions
    SELECT transaction_id, store_region, category,
           total_amount,
           ROUND(total_amount * 1.20, 2) AS total_ttc,  -- CORRIGÉ
           current_timestamp()
    FROM nessie.retailco.silver_transactions
    WHERE total_ttc < total_amount  -- lignes corrompues
""")

# 4. Validation (test data quality)
errors = spark.sql("""
    SELECT count(*) AS n_errors
    FROM nessie.retailco.silver_transactions
    WHERE total_ttc < total_amount
""").collect()[0]['n_errors']
print(f'\n✅ Test qualité : {errors} erreur(s) — {"PASSED" if errors == 0 else "FAILED"}')

# 5. Merger sur main
fix_hash = nessie('GET', '/trees/fix/silver-ttc-correction')['reference']['hash']
nessie('POST', '/trees/main/merge', {
    'fromRefName': 'fix/silver-ttc-correction',
    'fromHash': fix_hash
})
spark.conf.set('spark.sql.catalog.nessie.ref', 'main')
print('\n✅ Merge fix → main effectué')
print('\nÉtat final sur main :')
spark.sql('SELECT transaction_id, total_amount, total_ttc, ROUND(total_ttc/total_amount,2) AS ratio FROM nessie.retailco.silver_transactions').show()

---
## ⏱️ Débrief client (10 min)

### Questions obligatoires — rôle Client :

1. **"Comment vous avez su quel commit a introduit le bug ? En production, vous avez toujours accès à ce log ?"**
2. **"Le rollback a duré combien de temps ? Est-ce que les utilisateurs ont été impactés pendant ?"**
3. **"Comment vous assurez que ce type de bug ne passe plus jamais en production ?"**
4. **"Si deux équipes travaillent en parallèle sur deux branches — comment vous gérez les conflits ?"**

---

## Récapitulatif

| Compétence | Exercice | Transférable en prod |
|-----------|---------|---------------------|
| Isolation par branche | Créer feat/nessie-demo | Feature flags pour les données |
| Audit trail | Log Nessie + snapshots Iceberg | Traçabilité complète sans outil externe |
| Rollback instantané | Assign hash correct | RTO < 30s sur une table corrompue |
| Fix sur branche | fix/ branch + test + merge | Workflow PR pour les pipelines data |